# GBIS 데이터 불러오기 (팀원용)

이 노트북은 개발 환경 설정 없이 GBIS 데이터를 Google Colab에서 DataFrame으로 불러옵니다.

처음 한 번만 다음 작업을 해주세요.

1. Colab 왼쪽의 **열쇠(Secrets)** 아이콘을 누릅니다.
2. 이름이 `GBIS_API_KEY`인 새 보안 비밀을 만들고 전달받은 개인 키를 입력합니다.
3. 이 노트북에서 해당 보안 비밀에 대한 **Notebook access**를 켭니다.
4. 상단 메뉴에서 **런타임 → 모두 실행**을 누릅니다.

최초 실행은 전체 이력을 내려받으므로 시간이 걸릴 수 있습니다. 이후에는 Google Drive의 캐시를 복원하고 새 데이터만 받습니다.

In [1]:
# 아래 값은 특별한 경우가 아니면 그대로 사용하세요.
API_BASE_URL = "https://161.33.212.6"  # @param {type:"string"}
DRIVE_CACHE_PATH = "/content/drive/MyDrive/GBIS/gbis_api_cache.sqlite3"  # @param {type:"string"}
ROUTE_IDS = ""  # @param {type:"string"}
HISTORY_FROM = ""  # @param {type:"string"}

import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

try:
    API_KEY = userdata.get("GBIS_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "왼쪽 열쇠 아이콘에서 GBIS_API_KEY를 만들고 Notebook access를 켜주세요."
    ) from exc
if not API_KEY:
    raise RuntimeError("Colab Secrets의 GBIS_API_KEY가 비어 있습니다.")

REPO_DIR = Path("/content/gbis_team_repo")
REPO_URL = "https://github.com/khuda-data/10th-toy-team4.git"
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO_DIR / "requirements-client.txt"),
    ],
    check=True,
)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

drive.mount("/content/drive")
DRIVE_CACHE = Path(DRIVE_CACHE_PATH)
LOCAL_CACHE = Path("/content/gbis_api_cache.sqlite3")
DRIVE_CACHE.parent.mkdir(parents=True, exist_ok=True)
CACHE_RESTORED = DRIVE_CACHE.is_file()
if CACHE_RESTORED:
    shutil.copy2(DRIVE_CACHE, LOCAL_CACHE)
    print("✅ Google Drive에서 기존 캐시를 복원했습니다.")
else:
    print("ℹ️ 첫 실행입니다. 서버의 전체 이력을 내려받습니다.")

Mounted at /content/drive
ℹ️ 첫 실행입니다. 서버의 전체 이력을 내려받습니다.


In [2]:
from IPython.display import display
from gbis_client import GBISApiCache

requested_route_ids = tuple(
    value.strip() for value in ROUTE_IDS.split(",") if value.strip()
)
cache = GBISApiCache(
    base_url=API_BASE_URL,
    api_key=API_KEY,
    cache_path=LOCAL_CACHE,
)
history_counts = {}
try:
    print("1/4 노선 목록을 최신화합니다.")
    cache.refresh_routes()
    routes_df = cache.routes_df()
    available_route_ids = set(routes_df["route_id"].astype(str))
    if requested_route_ids:
        missing_route_ids = set(requested_route_ids) - available_route_ids
        if missing_route_ids:
            raise ValueError(f"서버에 없는 route_id입니다: {sorted(missing_route_ids)}")
        route_ids = requested_route_ids
    else:
        route_ids = tuple(str(value) for value in routes_df["route_id"].tolist())

    print("2/4 정류장 정보를 최신화합니다.")
    station_count_by_route = dict(zip(routes_df["route_id"].astype(str), routes_df["station_count"]))
    station_counts = {
        route_id: cache.refresh_stations(route_id)
        for route_id in route_ids
        if int(station_count_by_route.get(route_id, 0)) > 0
    }

    print("3/4 최신 차량 위치를 갱신합니다.")
    cache.refresh_latest()

    print("4/4 차량 위치 이력을 동기화합니다.")
    for index, route_id in enumerate(route_ids, 1):
        mode = "증분" if CACHE_RESTORED else "최초 전체"
        print(f"  [{index}/{len(route_ids)}] {route_id}: {mode} 동기화 중...")
        history_counts[route_id] = cache.refresh_full_history(route_id)

    routes_df = cache.routes_df()
    stations_df = cache.stations_df()
    latest_df = cache.latest_locations_df()
    history_df = cache.history_df(from_at=HISTORY_FROM or None)
    cache_status_df = cache.cache_status_df()
    if requested_route_ids:
        stations_df = stations_df[stations_df["route_id"].isin(route_ids)].reset_index(drop=True)
        latest_df = latest_df[latest_df["route_id"].isin(route_ids)].reset_index(drop=True)
        history_df = history_df[history_df["route_id"].isin(route_ids)].reset_index(drop=True)
finally:
    cache.close()
    if LOCAL_CACHE.is_file():
        shutil.copy2(LOCAL_CACHE, DRIVE_CACHE)
        print(f"✅ 캐시를 Google Drive에 저장했습니다: {DRIVE_CACHE}")

print("\n동기화 완료")
print(f"- routes_df: {len(routes_df):,}행")
print(f"- stations_df: {len(stations_df):,}행")
print(f"- latest_df: {len(latest_df):,}행")
print(f"- history_df: {len(history_df):,}행")
display(history_df.head())

1/4 노선 목록을 최신화합니다.
2/4 정류장 정보를 최신화합니다.
3/4 최신 차량 위치를 갱신합니다.
4/4 차량 위치 이력을 동기화합니다.
  [1/17] 200000104: 최초 전체 동기화 중...
  [2/17] 204000057: 최초 전체 동기화 중...
  [3/17] 218000010: 최초 전체 동기화 중...
  [4/17] 219000013: 최초 전체 동기화 중...
  [5/17] 219000016: 최초 전체 동기화 중...
  [6/17] 222000074: 최초 전체 동기화 중...
  [7/17] 222000075: 최초 전체 동기화 중...
  [8/17] 222000209: 최초 전체 동기화 중...
  [9/17] 228000174: 최초 전체 동기화 중...
  [10/17] 229000311: 최초 전체 동기화 중...
  [11/17] 232000090: 최초 전체 동기화 중...
  [12/17] 234000309: 최초 전체 동기화 중...
  [13/17] 234000878: 최초 전체 동기화 중...
  [14/17] 234001243: 최초 전체 동기화 중...
  [15/17] 234001245: 최초 전체 동기화 중...
  [16/17] 234001695: 최초 전체 동기화 중...
  [17/17] 234001736: 최초 전체 동기화 중...
✅ 캐시를 Google Drive에 저장했습니다: /content/drive/MyDrive/GBIS/gbis_api_cache.sqlite3

동기화 완료
- routes_df: 17행
- stations_df: 1,220행
- latest_df: 127행
- history_df: 283,771행


,route_id,vehicle_id,observed_at,query_time,plate_no,route_type_code,station_id,station_seq,station_name,remaining_seats,crowded,low_plate,state_code,tagless_code,cached_at_utc
0,219000013,218000030,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아3485,11,219000561,52,대화역(중),39,1,0,0,0,2026-08-11T17:05:29+00:00
1,219000013,218000160,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1117,11,100000034,27,광화문역6번출구.광화문빌딩,41,1,0,2,1,2026-08-11T17:05:29+00:00
2,219000013,218000161,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1118,11,277103099,26,경복궁역(경유),43,1,0,2,1,2026-08-11T17:05:29+00:00
3,219000013,218000162,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1120,11,112000012,23,연세대앞(중),33,1,0,2,1,2026-08-11T17:05:29+00:00
4,219000013,218000165,2026-08-03T13:22:00+09:00,2026-08-03 13:22:09.124,경기73아1134,11,277103096,21,증산교교차로(경유),34,1,0,0,1,2026-08-11T17:05:29+00:00


## 사용할 수 있는 DataFrame

- `routes_df`: 노선 목록과 수집 범위
- `stations_df`: 노선별 정류장 순서와 위치
- `latest_df`: 현재 운행 차량별 최신 위치와 잔여좌석
- `history_df`: 전체 또는 지정 시각 이후의 차량 위치 이력
- `cache_status_df`: 데이터별 마지막 갱신 완료 시각

예를 들어 잔여좌석이 0인 기록은 `history_df[history_df["remaining_seats"] == 0]`으로 확인할 수 있습니다.

In [10]:
import pandas as pd


history_df['observed_at'] = pd.to_datetime(history_df['observed_at'])


rush_hour_df = history_df[
    (history_df['observed_at'].dt.hour >= 7) &
    (history_df['observed_at'].dt.hour < 9)
].copy()


columns_to_keep = [
    'observed_at', 'route_id', 'vehicle_id',
    'station_seq', 'station_name', 'remaining_seats'
]


rush_hour_summary = rush_hour_df[columns_to_keep]
rush_hour_summary.head()


df_sorted = rush_hour_summary.sort_values(by=['vehicle_id', 'observed_at']).reset_index(drop=True)

df_sorted['seats_5_stops_ago'] = df_sorted.groupby('vehicle_id')['remaining_seats'].shift(5)


model_df = df_sorted.dropna(subset=['seats_5_stops_ago']).copy()


print("=== 모델 학습용 데이터 준비 완료 ===")
model_df[['vehicle_id', 'station_name', 'station_seq', 'seats_5_stops_ago', 'remaining_seats']].head()

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


X = model_df[['seats_5_stops_ago', 'station_seq']]
y = model_df['remaining_seats']


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)


y_pred = model.predict(X_val)
mae = mean_absolute_error(y_val, y_pred)

print(f" 모델 학습 성공!")
print(f" 평균 예측 오차(MAE): 약 {mae:.2f}좌석 차이")


results_df = X_val.copy()


results_df['5정거장_전_좌석'] = results_df['seats_5_stops_ago']


results_df['실제_도착시_좌석'] = y_val


results_df['AI_예측_좌석'] = y_pred.round().astype(int)


results_df['오차(석)'] = abs(results_df['실제_도착시_좌석'] - results_df['AI_예측_좌석'])

print("=== AI 예측 결과 vs 실제 결과 비교표 ===")
results_df[['5정거장_전_좌석', '실제_도착시_좌석', 'AI_예측_좌석', '오차(석)']].head(10)

=== 모델 학습용 데이터 준비 완료 ===
 모델 학습 성공!
 평균 예측 오차(MAE): 약 2.54좌석 차이
=== AI 예측 결과 vs 실제 결과 비교표 ===


,5정거장_전_좌석,실제_도착시_좌석,AI_예측_좌석,오차(석)
4651,43.0,11,40,29
5259,39.0,42,39,3
16248,25.0,22,20,2
16882,0.0,0,0,0
6511,44.0,25,29,4
9448,44.0,43,38,5
6765,40.0,40,40,0
36804,41.0,40,38,2
37074,40.0,40,40,0
3187,41.0,43,41,2
